In [11]:
# 프롬프트 엔지니어링, 파인튜닝의 차이
# zero-shot prompting, few-shot prompting, 전체 매개변수 미세조정(full Fine Turning)
# 성능(accuracy) 자원소모(학습시간, 파라메터 업데이트 비율)
# google/flan-t5-small( seq2seq 소형 모델)

In [12]:
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


### 데이터셋

In [22]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

100%|██████████| 25.7M/25.7M [00:00<00:00, 27.0MB/s]


Extracting files...
Path to dataset files: C:\Users\Playdata\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1


In [13]:
train_data = [
    {"text": "This movie was absolutely fantastic! The acting was superb.", "label": "positive"},
    {"text": "I wasted two hours of my life. Terrible plot and cheap CGI.", "label": "negative"},
    {"text": "A true masterpiece of cinema. Highly recommend it to everyone.", "label": "positive"},
    {"text": "Extremely boring and predictable. I fell asleep halfway through.", "label": "negative"},
    {"text": "The cinematography was beautiful, and the story was touching.", "label": "positive"},
    {"text": "Horrible acting and bad direction. Do not watch this film.", "label": "negative"},
    {"text": "Brilliant performance by the lead actor, a must-watch.", "label": "positive"},
    {"text": "The plot made no sense, and the characters were annoying.", "label": "negative"},
    {"text": "I loved the soundtrack and the emotional depth of this movie.", "label": "positive"},
    {"text": "Total waste of money. I regret watching this garbage.", "label": "negative"},
    {"text": "Very engaging storyline with great character development.", "label": "positive"},
    {"text": "Slow, dull, and lacks any real substance.", "label": "negative"},
    {"text": "An amazing adventure that kept me on the edge of my seat.", "label": "positive"},
    {"text": "A complete disappointment. The trailer was much better than the film.", "label": "negative"},
    {"text": "Wonderfully written and beautifully executed. A delight to watch.", "label": "positive"}
]

test_data = [
    {"text": "The movie was mediocre, but the ending was spectacular!", "label": "positive"},
    {"text": "Worst movie I have seen this year. Avoid at all costs.", "label": "negative"},
    {"text": "A refreshing and delightful comedy that had me laughing all night.", "label": "positive"},
    {"text": "The acting was flat and the script felt extremely forced.", "label": "negative"},
    {"text": "An absolute gem of a film with outstanding visuals.", "label": "positive"}
]

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size: {len(test_data)}")

Train dataset size: 15
Test dataset size: 5


## 모델 로드 및 모델 매개변수(Parameter) 분석
Hugging Face의 `transformers` 라이브러리를 활용하여 구글이 공개한 지시어 미세조정(Instruction-tuned) 모델인 `google/flan-t5-small`을 로드합니다.
또한 파인튜닝 전에 모델의 **총 매개변수(Total Parameters)** 와 **학습 가능한 매개변수(Trainable Parameters)** 를 확인합니다.

In [14]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [15]:
# _, param = next(iter(model.named_parameters()))
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

(76961152, 76961152, 100.0)

In [16]:
# 추론 수행 함수
def generate_prediction(prompt, model, tokenizer,max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
    return pred_text
# 평가 루프
def evaluate_model(prompt, tokenizer, test_data, prompt_type='zero_shot'):
    correct = 0; results=[]
    # few shot 예시
    few_shot_examples = (
        "Review: The cinematography was beautiful, and the story was touching.\nSentiment:positive\n\n"
        "Review: Horrible acting and bad direction. Do not watch this film.\nSentiment:negative\n\n"
        "Review: Brilliant performance by the lead actor, a must-watch.\nSentiment:positive\n\n"
    )
    for item in test_data:
        text = item['text']
        true_label = item['label']
        if prompt_type =='zero_shot':
            prompt=f'Review:{text}\nSentiment: Answer with either positive or negative."'
        elif prompt_type =='few_shot':
            prompt=few_shot_examples + f'Review:{text}\nSentiment:  Answer with either positive or negative."'
        else:
            prompt=f'Review:{text}\nSentiment:  Answer with either positive or negative."'
        pred_label = generate_prediction(prompt,model,tokenizer)
        # 긍정/부정 단어 클랜징
        if 'positive' in pred_label:
            cleaned_pred = 'positive'
        elif 'negative' in pred_label:
            cleaned_pred = 'negative'
        else:
            cleaned_pred = pred_label
        is_correct = (cleaned_pred == true_label)
        if is_correct:
            correct += 1
        results.append({
            'Text':text, 'True Label' : true_label, 
            'Pred Label':pred_label, 'Cleaned Pred' : cleaned_pred, 'Correct':is_correct
        })
    accuracy = correct / len(test_data)
    return accuracy, pd.DataFrame(results)

### Zero Shot Prompting

In [17]:
zero_shot_acc, zero_shot_df =  evaluate_model(model, tokenizer,test_data,'zero_shot')
print(f'zero shot acc : {zero_shot_acc}')
zero_shot_df

zero shot acc : 1.0


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The movie was mediocre, but the ending was spe...",positive,positive,positive,True
1,Worst movie I have seen this year. Avoid at al...,negative,negative,negative,True
2,A refreshing and delightful comedy that had me...,positive,positive,positive,True
3,The acting was flat and the script felt extrem...,negative,negative,negative,True
4,An absolute gem of a film with outstanding vis...,positive,positive,positive,True


### Few-Shot Prompting

In [18]:
fes_shot_acc, fes_shot_df =  evaluate_model(model, tokenizer,test_data,'few_shot')
print(f'fes shot acc : {fes_shot_acc}')
fes_shot_df

fes shot acc : 1.0


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The movie was mediocre, but the ending was spe...",positive,positive,positive,True
1,Worst movie I have seen this year. Avoid at al...,negative,negative,negative,True
2,A refreshing and delightful comedy that had me...,positive,positive,positive,True
3,The acting was flat and the script felt extrem...,negative,negative,negative,True
4,An absolute gem of a film with outstanding vis...,positive,positive,positive,True


### 전체 매개변수 파인튜닝(Full Fine Tuning)

In [19]:
# 1. 데이터 토크나이징 함수 - 전처리
def preprocess_fuction(example):
    # T5모델에 맞게 토큰화
    inputs = [ f"Review: {text}\nSentiment: Answer with either positive or negative."  for text in example['text'] ]
    model_input = tokenizer(inputs,max_length=128, truncation=True)
    # 라벨을 토큰화
    lables = tokenizer(text_target=example['label'],max_length=128, truncation=True)
    model_input['labels'] = lables['input_ids']
    return model_input
# 2. DataSet객체
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# 3. 토큰화 매핑()
tokenized_train = train_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [20]:
# training Arguments 설정
training_args = Seq2SeqTrainingArguments(
    output_dir='./t5_sentiment_results',
    eval_strategy='epoch',
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=2
)
# Data Collator, Trainer 객체
data_collator = DataCollatorForSeq2Seq(tokenizer,model=model)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,train_dataset=tokenized_train, eval_dataset=tokenized_test,
    processing_class=tokenizer, data_collator = data_collator
)
# 학습시간 측정
start_time = time.time() 
trainer.train()
training_time = time.time() - start_time
print(f'Fine-turning completed : {training_time:.2f} seconds') 

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.123075,0.015057
2,0.002622,0.006754
3,0.000927,0.003861
4,0.000124,0.002841
5,0.000479,0.002552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fine-turning completed : 8.55 seconds


In [21]:
ft_acc, ft_df = evaluate_model(model, tokenizer, test_data, "fine_tuned")
print(f"Fine-tuned Model Accuracy: {ft_acc:.2%}")
print(ft_df[["Text", "True Label", "Pred Label", "Correct"]].to_string())

Fine-tuned Model Accuracy: 100.00%
                                                                 Text True Label Pred Label  Correct
0             The movie was mediocre, but the ending was spectacular!   positive   positive     True
1              Worst movie I have seen this year. Avoid at all costs.   negative   negative     True
2  A refreshing and delightful comedy that had me laughing all night.   positive   positive     True
3           The acting was flat and the script felt extremely forced.   negative   negative     True
4                 An absolute gem of a film with outstanding visuals.   positive   positive     True
